# AI Probes vs Other Attackers: Visual Contrast

Do IPs that try AI-tooling usernames (`claude`, `ollama`, `gpu`, ...) differ from other attackers? Key idea for this analysis:
- enrich every `src_ip` with country and ASN, and check it against public blocklists
- then screen 17 per-IP properties with chi-square, Fisher and Mann-Whitney tests under Benjamini-Hochberg FDR control.
- Method and caveats: `../README.md`, section "Do AI probes differ from other attackers?".

Exploration only. Logic lives in `../ai_contrast.py`, `../../common/stats.py`, `../../common/enrich.py`, `../../common/threatintel.py` and `../charts.py`. This notebook only calls into them.

**Setup**: same "Python (bear-trap analysis)" kernel as `eda.ipynb`, see `../README.md`. The first run downloads the DB-IP Lite databases (~11 MB) into `../../common/outputs/enrichment/`, and the two blocklists (~150 KB) into `../../common/outputs/threatintel/`. Delete a cached list to refresh it.

**How to read it**
- Group a = **AI probe**: tried at least one AI-tooling username. Group b = **matched other**: didn't. Both groups only include IPs that tried >= `MIN_USERNAMES` distinct usernames, because AI probes are mostly big dictionaries.
- Judge significance by `q_bh` / `significant_bh`, and size by `effect_size`. With hundreds of IPs, a negligible difference can still be significant.

In [ ]:
from IPython.display import Image, display

from honeypot.analysis.common.enrich import load_enrichment
from honeypot.analysis.common.extract import get_sessions
from honeypot.analysis.common.stats import category_breakdown
from honeypot.analysis.common.threatintel import listing_coverage, load_listings
from honeypot.analysis.network.ai_contrast import (
    DEFAULT_MIN_USERNAMES,
    GROUP_COLUMN,
    NUMERIC_PROPERTIES,
    ai_probe_profiles,
    compare_ai_probes,
    one_ip_per_cluster,
    without_top_ai_level,
)
from honeypot.analysis.network.charts import (
    plot_ai_probe_breakdown,
    plot_ai_probe_distributions,
    plot_ai_probe_effect_sizes,
)
from honeypot.analysis.network.graph import (
    assign_clusters,
    build_credential_graph,
    build_ip_similarity_graph,
)

## Load and enrich

`"offline"` reads the most recent local snapshot; `"live"` pulls fresh from BigQuery. Enrichment adds `country`, `asn`, `as_org` and `as_label` per IP. The null counts show how many IPs DB-IP couldn't place. `listings` flags IPs on the blocklist.de SSH list and FireHOL level1, as downloaded on the date in the cached filename, not at attack time (see `../../common/threatintel.py`).

In [ ]:
MODE = "offline"

events = get_sessions(MODE)
enrichment = load_enrichment(events)
listings = load_listings(events)
enrichment.null_count()

## The size-matched population

One row per IP. Raise `MIN_USERNAMES` for a tighter match on dictionary size, at the cost of fewer IPs.

In [ ]:
MIN_USERNAMES = DEFAULT_MIN_USERNAMES

profiles = ai_probe_profiles(
    events, enrichment=enrichment, min_usernames=MIN_USERNAMES, listings=listings
)
profiles.group_by(GROUP_COLUMN).len()

## The screen

One row per property, sorted by `q_bh`:
- `n_a` / `n_b`: AI-probe / matched IPs with a value for that property.
- `detail`: the level most over-represented among AI probes, or the two medians.
- `p_holm` / `p_bonferroni`: the stricter family-wise corrections, for comparison.

In [ ]:
comparison = compare_ai_probes(profiles)
comparison

## Is it one campaign?

IPs from one campaign aren't independent, and a single large one can produce most of the categorical hits. This removes every IP in the ASN most over-represented among AI probes, from **both** groups, and reruns the screen.

A second check counts each credential cluster (`graph.assign_clusters`: IPs that share a distinctive part of their wordlist) once per group, keeping its busiest IP. If an effect shrinks here too, it came from a few campaigns spread over many IPs rather than from many independent attackers.

In the chart below, a long gray bar means a property's effect changes a lot between these scenarios. A missing dot means the property became untestable in that scenario (e.g. `as_label` collapses to a single level once the ASN is removed).

In [ ]:
top_asn, without_top = without_top_ai_level(profiles, "as_label")
comparisons = {"all matched IPs": comparison}
if top_asn is not None:
    comparisons[f"excluding {top_asn}"] = compare_ai_probes(without_top)
    print(f"Excluded {profiles.height - without_top.height} IPs in {top_asn}")
clusters = assign_clusters(build_ip_similarity_graph(build_credential_graph(events)))
by_cluster = one_ip_per_cluster(profiles, clusters)
comparisons["one IP per cluster"] = compare_ai_probes(by_cluster)
print(f"One IP per cluster: {profiles.height} -> {by_cluster.height} IPs")

In [ ]:
Image(plot_ai_probe_effect_sizes(comparisons))

## Already on a blocklist?

Share of each group's IPs on each list. A low share for AI probes would mean public feeds don't know this traffic yet. blocklist.de is partly fed by other honeypots, so its hits are partly circular.

In [ ]:
listing_coverage(
    profiles.select("src_ip", GROUP_COLUMN).join(listings, on="src_ip"),
    GROUP_COLUMN,
)

## Categorical drill-downs

The exact levels each test compared (rare levels pooled into `other`), as the share of each group's IPs. Shares rather than counts, because the matched group is about 5x larger.

In [ ]:
for column in ["country", "as_label", "hassh"]:
    display(
        Image(
            plot_ai_probe_breakdown(
                category_breakdown(profiles, GROUP_COLUMN, column), column
            )
        )
    )

## Numeric distributions

Box plots on a symlog axis (heavy-tailed counts that include zeros). The volume properties (`distinct_passwords`, `credential_attempts`, `sessions`) track `distinct_usernames`. If AI probes still try more usernames after matching, these differences partly reflect dictionary size, not anything AI-specific.

In [ ]:
Image(plot_ai_probe_distributions(profiles, NUMERIC_PROPERTIES))

## Exploration

Ad hoc from here. Open threads:
- Rerun `without_top_ai_level` on `hassh` instead of `as_label`. Does the same tool also run from other ASNs?
- `active_days` changes sign once the top ASN is removed. Was the campaign a short burst?